In [0]:
# =============================================================================
# CAMADA SILVER - Limpeza e padronização dos dados
# =============================================================================
# Objetivo: Ler as tabelas Bronze, aplicar limpeza, padronização de tipos,
# remoção de duplicatas e valores nulos críticos, e salvar as tabelas Silver
# prontas para modelagem e análise.
# =============================================================================

from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, IntegerType, TimestampType

# =============================================================================
# 1. ORDERS - Pedidos
# =============================================================================
# Leitura da tabela Bronze de pedidos
orders = spark.table("workspace.default.bronze_orders")

# Remove colunas de metadados da camada Bronze, não são necessárias na Silver
orders = orders.drop("_source_file", "_ingestion_date")

# Converte colunas de data para o tipo Timestamp correto
# Na Bronze vieram como string por limitação da inferência do CSV
date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]
for col in date_cols:
    orders = orders.withColumn(col, F.to_timestamp(col))

# Remove pedidos sem status definido (registro inválido)
orders = orders.filter(F.col("order_status").isNotNull())

# Remove duplicatas por order_id, que deve ser único
orders = orders.dropDuplicates(["order_id"])

# Salva como tabela Silver
orders.write.format("delta").mode("overwrite").saveAsTable("workspace.default.silver_orders")
print(f"✓ silver_orders salva com {orders.count()} registros")

# =============================================================================
# 2. ORDER ITEMS - Itens dos pedidos
# =============================================================================
items = spark.table("workspace.default.bronze_order_items")
items = items.drop("_source_file", "_ingestion_date")

# Garante que preço e frete são do tipo numérico decimal
items = items.withColumn("price", F.col("price").cast(DoubleType()))
items = items.withColumn("freight_value", F.col("freight_value").cast(DoubleType()))

# Remove itens com preço nulo ou negativo (registros inválidos)
items = items.filter(F.col("price").isNotNull() & (F.col("price") >= 0))

# Remove duplicatas pela chave composta (pedido + sequência do item)
items = items.dropDuplicates(["order_id", "order_item_id"])

items.write.format("delta").mode("overwrite").saveAsTable("workspace.default.silver_order_items")
print(f"✓ silver_order_items salva com {items.count()} registros")

# =============================================================================
# 3. ORDER PAYMENTS - Pagamentos
# =============================================================================
payments = spark.table("workspace.default.bronze_order_payments")
payments = payments.drop("_source_file", "_ingestion_date")

# Garante tipo numérico no valor do pagamento
payments = payments.withColumn("payment_value", F.col("payment_value").cast(DoubleType()))
payments = payments.withColumn("payment_installments", F.col("payment_installments").cast(IntegerType()))

# Remove pagamentos com valor nulo ou negativo
payments = payments.filter(F.col("payment_value").isNotNull() & (F.col("payment_value") >= 0))

payments = payments.dropDuplicates(["order_id", "payment_sequential"])

payments.write.format("delta").mode("overwrite").saveAsTable("workspace.default.silver_order_payments")
print(f"✓ silver_order_payments salva com {payments.count()} registros")

# =============================================================================
# 4. PRODUCTS - Produtos
# =============================================================================
products = spark.table("workspace.default.bronze_products")
products = products.drop("_source_file", "_ingestion_date")

# Remove produtos sem categoria definida (não contribuem para análise por categoria)
products = products.filter(F.col("product_category_name").isNotNull())

# Remove duplicatas por product_id
products = products.dropDuplicates(["product_id"])

products.write.format("delta").mode("overwrite").saveAsTable("workspace.default.silver_products")
print(f"✓ silver_products salva com {products.count()} registros")

# =============================================================================
# 5. CUSTOMERS - Clientes
# =============================================================================
customers = spark.table("workspace.default.bronze_customers")
customers = customers.drop("_source_file", "_ingestion_date")

# Padroniza siglas de estado para maiúsculo (evita inconsistências como "sp" vs "SP")
customers = customers.withColumn("customer_state", F.upper(F.col("customer_state")))

# Remove clientes sem estado definido
customers = customers.filter(F.col("customer_state").isNotNull())

customers = customers.dropDuplicates(["customer_id"])

customers.write.format("delta").mode("overwrite").saveAsTable("workspace.default.silver_customers")
print(f"✓ silver_customers salva com {customers.count()} registros")

# =============================================================================
# 6. CATEGORY TRANSLATION - Tradução das categorias
# =============================================================================
translation = spark.table("workspace.default.bronze_category_translation")
translation = translation.drop("_source_file", "_ingestion_date")

# Remove categorias sem tradução disponível
translation = translation.filter(F.col("product_category_name_english").isNotNull())

translation = translation.dropDuplicates(["product_category_name"])

translation.write.format("delta").mode("overwrite").saveAsTable("workspace.default.silver_category_translation")
print(f"✓ silver_category_translation salva com {translation.count()} registros")

# =============================================================================
# 7. SELLERS - Vendedores
# =============================================================================
sellers = spark.table("workspace.default.bronze_sellers")
sellers = sellers.drop("_source_file", "_ingestion_date")

# Padroniza sigla do estado para maiúsculo
sellers = sellers.withColumn("seller_state", F.upper(F.col("seller_state")))

sellers = sellers.filter(F.col("seller_state").isNotNull())

sellers = sellers.dropDuplicates(["seller_id"])

sellers.write.format("delta").mode("overwrite").saveAsTable("workspace.default.silver_sellers")
print(f"✓ silver_sellers salva com {sellers.count()} registros")

# =============================================================================
# 8. ORDER REVIEWS - Avaliações dos pedidos
# =============================================================================
# Tabela adicionada para suportar a Pergunta 6 do objetivo (relação entre
# avaliação e valor gasto), mantendo a coerência da arquitetura medalhão.
# A análise final deve consumir da Silver, nunca da Bronze diretamente.
# =============================================================================

reviews = spark.table("workspace.default.bronze_order_reviews")
reviews = reviews.drop("_source_file", "_ingestion_date")

# O campo review_score apresenta valores malformados na origem (timestamps
# gravados no lugar de inteiros, provável erro de parsing do CSV original).
# try_cast converte valores inválidos para null em vez de lançar exceção.
reviews = reviews.withColumn(
    "review_score",
    F.expr("try_cast(review_score as int)")
)

# Mantém apenas notas dentro do domínio válido da escala (1 a 5)
reviews = reviews.filter(
    F.col("review_score").isNotNull() &
    F.col("review_score").between(1, 5)
)

# Converte datas de avaliação para timestamp
for col in ["review_creation_date", "review_answer_timestamp"]:
    reviews = reviews.withColumn(col, F.to_timestamp(col))

# review_id é a chave única da tabela
reviews = reviews.dropDuplicates(["review_id"])

reviews.write.format("delta").mode("overwrite").saveAsTable("workspace.default.silver_order_reviews")
print(f"✓ silver_order_reviews salva com {reviews.count()} registros")